# Raw kinetics data

Load the local BRENDA archive and SABIO-RK JSON pages. Inspect original values, comments, linked proteins and publications, then review the exploratory BRENDA coverage tables in memory. No tables or datasets are exported.

Change the selected EC, parameter entry or SABIO ID below. Overview tables show three rows; selected entries show full text. `brenda_raw`, `brenda_kinetic_raw`, `sabio_pages` and `sabio_records` retain the source content for further inspection.

In [1]:
from pathlib import Path
import json
import math
import re
import tarfile

import ijson
import pandas as pd
from IPython.display import display

ROOT = next(
    parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "assets/tools/proteolysis_data_manager").is_dir()
)
RAW = ROOT / "data/proteolysis/kinetics_datasets/raw"
PREVIEW_ROWS = 3
BRENDA_EC = "3.4.21.4"
BRENDA_KINETIC_FIELD = "turnover_number"  # Also km_value or kcat_km_value.
BRENDA_ENTRY_INDEX = 0
SABIO_ENTRY_ID = 62365
SHOW_FULL_RECORDS = False
KINETIC_FAMILIES = ("turnover_number", "km_value", "kcat_km_value")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

## 1. Source files

Read the existing files listed in the acquisition manifest.

In [2]:
raw_manifest = json.loads((RAW / "manifest.json").read_text())
source_files = pd.DataFrame([
    {"source": source, "filename": filename, "bytes": (RAW / source / filename).stat().st_size}
    for source in ("brenda", "sabio")
    for filename in raw_manifest["sources"][source]["files"]
])
display(source_files)

,source,filename,bytes
0,brenda,brenda_2026.1_json.tar.gz,83199555
1,sabio,page_000001.json,8529332
2,sabio,page_000002.json,8521768
3,sabio,page_000003.json,7992307
4,sabio,page_000004.json,2630785


## 2. BRENDA: raw entries

Stream the archive once. Keep the complete selected EC block and the original kinetic entries for all EC `3.4.*`, without an organism filter. Provenance columns identify each original row; the source fields `value`, `comment`, `proteins` and `references` are unchanged.

In [3]:
brenda_filename = next(iter(raw_manifest["sources"]["brenda"]["files"]))
brenda_raw = None
brenda_rows = []
brenda_ec_count = 0
with tarfile.open(RAW / "brenda" / brenda_filename, "r:*") as archive:
    members = [member for member in archive.getmembers() if member.isfile() and member.name.endswith(".json")]
    if len(members) != 1:
        raise ValueError("Expected exactly one BRENDA JSON member")
    with archive.extractfile(members[0]) as source_json:
        for ec, block in ijson.kvitems(source_json, "data"):
            if ec == BRENDA_EC:
                brenda_raw = block
            if not ec.startswith("3.4."):
                continue
            brenda_ec_count += 1
            for family in KINETIC_FAMILIES:
                for index, entry in enumerate(block.get(family) or []):
                    brenda_rows.append({
                        "ec_number": ec, "parameter_family": family, "source_index": index,
                        "source_record_id": f"data/{ec}/{family}/{index}", **entry,
                    })
if brenda_raw is None:
    raise KeyError(f"EC {BRENDA_EC} is absent from the source archive")
brenda_kinetic_raw = pd.DataFrame(brenda_rows, dtype=object)
print("EC blocks:", brenda_ec_count, "| Kinetic parameter entries:", len(brenda_kinetic_raw))
print("Selected EC:", BRENDA_EC, "|", brenda_raw.get("recommended_name"))
for family in KINETIC_FAMILIES:
    entries = pd.DataFrame(brenda_raw.get(family) or [], dtype=object)
    print(family, "| Entries:", len(entries))
    display(entries.head(PREVIEW_ROWS))

EC blocks: 542 | Kinetic parameter entries: 26697
Selected EC: 3.4.21.4 | trypsin
turnover_number | Entries: 386


,proteins,references,comment,value
0,[1],[13],cationic trypsin 2,1 {Nalpha-benzoyl-DL-Arg-p-nitroanilide}
1,[1],[13],anionic trypsin,0.355 {Nalpha-benzoyl-DL-Arg-p-nitroanilide}
2,[1],[13],cationic trypsin 1,0.473 {Nalpha-benzoyl-DL-Arg-p-nitroanilide}


km_value | Entries: 413


,proteins,references,comment,value
0,[1],[13],cationic trypsin 1,0.197 {Nalpha-benzoyl-DL-Arg-p-nitroanilide}
1,[1],[13],cationic trypsin 2,0.244 {Nalpha-benzoyl-DL-Arg-p-nitroanilide}
2,[1],[13],anionic trypsin,0.619 {Nalpha-benzoyl-DL-Arg-p-nitroanilide}


kcat_km_value | Entries: 76


,proteins,references,comment,value
0,[101],[234],"mutant Y71L, pH 8.0, 37°C",0.0054 {benzyloxycarbonylglycyl-L-prolyl-L-arginine-4-methylcoumaryl-7-amide}
1,[101],[234],"wild-type enzyme, pH 8.0, 37°C",0.0419 {benzyloxycarbonylglycyl-L-prolyl-L-arginine-4-methylcoumaryl-7-amide}
2,[109],[249],"pH 8.0, 37°C",144700 {benzyloxycarbonyl-Phe-Ser-Arg-7-amido-4-methylcoumarin}


### Selected kinetic entry, proteins and publications

The entry index is zero-based. Resolve only its explicit protein/reference IDs inside the selected EC block. Every linked record is shown; unresolved IDs remain visible as missing records.

In [4]:
brenda_selected = brenda_raw[BRENDA_KINETIC_FIELD][BRENDA_ENTRY_INDEX]
print(f"Source: {brenda_filename}#/data/{BRENDA_EC}/{BRENDA_KINETIC_FIELD}/{BRENDA_ENTRY_INDEX}")
display(pd.DataFrame([brenda_selected], dtype=object))
brenda_linked = {}
for link_field, dictionary_field in (("proteins", "protein"), ("references", "reference")):
    dictionary = brenda_raw.get(dictionary_field) or {}
    linked_rows = []
    for identifier in dict.fromkeys(brenda_selected.get(link_field) or []):
        source_record = dictionary.get(str(identifier))
        linked_rows.append({
            "source_id": str(identifier), "record_found": source_record is not None,
            **(source_record or {}),
        })
    brenda_linked[dictionary_field] = pd.DataFrame(linked_rows, dtype=object)
    print(dictionary_field)
    display(brenda_linked[dictionary_field])

Source: brenda_2026.1_json.tar.gz#/data/3.4.21.4/turnover_number/0


,proteins,references,comment,value
0,[1],[13],cationic trypsin 2,1 {Nalpha-benzoyl-DL-Arg-p-nitroanilide}


protein


,source_id,record_found,id,organism,references,comment
0,1,True,1,Gallus gallus,[13],


reference


,source_id,record_found,id,title,authors,journal,year,pages,vol,pmid
0,13,True,13,"Purification and partial characterization of the pancreatic proteolytic enzymes trypsin, chymotrypsin and elastase from the chicken","[Guyonnet, V., Tluscik, F., Long, P.L., Polanowski, A., Travis, J.]",J. Chromatogr.,1999,217-225,852,10480246


### Reaction descriptions in the same EC block

These are previews of source context. Shared EC, protein or publication IDs alone do not assign these descriptions to the selected kinetic measurement. The complete source lists remain in `brenda_raw`.

In [5]:
for field in ("reaction", "substrates_products", "natural_substrates_products"):
    entries = pd.DataFrame(brenda_raw.get(field) or [], dtype=object)
    print(field, "| Entries:", len(entries))
    display(entries.head(PREVIEW_ROWS))

reaction | Entries: 2


,proteins,references,comment,value
0,[7],[1],mechanism,"preferential cleavage: Arg-/-, Lys-/-"
1,[7],[242],"catalytic mechanism involving both His57 and Asp102, overview","preferential cleavage: Arg-/-, Lys-/-"


substrates_products | Entries: 449


,proteins,references,comment,value
0,"[21, 34]","[7, 10]",Nalpha-benzoyl-L-Arg-p-nitroanilide,Nalpha-benzoyl-DL-Arg-p-nitroanilide + H2O = ?
1,"[1, 6, 7, 9]","[2, 3, 13, 17, 21, 30]",,Nalpha-benzoyl-DL-Arg-p-nitroanilide + H2O = ?
2,[100],[237],,Nalpha-benzoyl-DL-arginine 4-nitroanilide + H2O = Nalpha-benzoyl-DL-Arg + 4-nitroaniline


natural_substrates_products | Entries: 35


,proteins,references,comment,value
0,[2],[158],"trypsin is the major endopeptidase regulating N-terminal truncated sex peptide DrmSP8-36 in hemolymph, the trypsin cleavage site R7-K8 is essential for release of the C-terminal fragment SP8-36 from sperm tail, the protein is cleaved more rapidly and completely in mated female hemolymph",DrmSP8-36 + H2O = ?
1,[4],[73],,Gelatin + H2O = ?
2,[4],[264],,beta-casein + H2O = ?


## 3. BRENDA: exploratory cleavage-information coverage

Count source parameter entries, not independent experiments or unique sequence windows. The four categories are exclusive:

| Category | Reading rule |
|---|---|
| `reported_bond_or_position` | Comment names a cleavage bond, position or motif, including ambiguous motifs such as `P-X`. |
| `relative_site_label` | Comment identifies a first/second site without coordinates. |
| `reporter_construct` | Substrate ends in one of the listed terminal amide reporter names; the reporter linkage is inferred from the construction. |
| `unresolved` | None of these rules matched. The publication may contain further information. |

Rules use the kinetic entry's own label and comment. A keyword about enzyme activation does not establish a substrate cut. EDDnp/FRET quenchers alone do not identify an internal cut. These counts do not establish complete P4–P1/P1′–P4′ windows. The rule definitions below can be expanded for inspection.

In [6]:
STATUS_ORDER = (
    "reported_bond_or_position", "relative_site_label", "reporter_construct", "unresolved",
)
AA3_PATTERN = "Ala|Arg|Asn|Asp|Cys|Gln|Glu|Gly|His|Ile|Leu|Lys|Met|Phe|Pro|Ser|Thr|Trp|Tyr|Val"
RESIDUE_PATTERN = rf"(?:(?:{AA3_PATTERN})|[ACDEFGHIKLMNPQRSTVWY])"
POSITION_PAIR_PATTERN = rf"{RESIDUE_PATTERN}\d+\s*[-–]\s*{RESIDUE_PATTERN}\d+"
EXPLICIT_SITE_PATTERN = re.compile("|".join([
    rf"\bcleavage\s+(?:at\s+)?(?:{POSITION_PAIR_PATTERN}|(?:{AA3_PATTERN})\d+)\b",
    rf"\bscissile\s+bond\s+{POSITION_PAIR_PATTERN}\b",
    rf"\b{POSITION_PAIR_PATTERN}\s+bond\b",
    r"\bcleavage\s+of\s+(?:the\s+)?[ACDEFGHIKLMNPQRSTVWYX]-[ACDEFGHIKLMNPQRSTVWYX]\s+bond\b",
]), re.IGNORECASE)
RELATIVE_SITE_PATTERN = re.compile(r"\b(?:first|second) cleavage site\b", re.IGNORECASE)
REPORTER_SUFFIXES = (
    "7-amido-4-carbamoylmethylcoumarin", "7-amido-4-fluoromethylcoumarin",
    "7-amino-4-methylcoumarin", "7-amido-4-methylcoumarin",
    "4-methylcoumarin-7-amide", "4-methyl-coumaryl-7-amide",
    "4-methoxy-2-naphthylamide", "2-naphthylamide",
    "4-nitroanilide", "p-nitroanilide", "nitroanilide", "pNA", "AMC", "AFC", "ACC", "MNA",
)
# EDDnp/EDnp and other FRET quenchers do not identify an internal cut by themselves.
REPORTER_PATTERN = re.compile(
    r"(?:-|\s)(" + "|".join(re.escape(item) for item in sorted(REPORTER_SUFFIXES, key=len, reverse=True)) + r")$",
    re.IGNORECASE,
)
NUMBER_PATTERN = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"
POINT_PATTERN = re.compile(rf"({NUMBER_PATTERN})(?:\s*(?:±|\+/-)\s*({NUMBER_PATTERN}))?")


def classify_source_entry(substrate, comment):
    """Classify source wording without predicting a cleavage position.

    :param str substrate: Original substrate label from the value braces.
    :param str comment: Original kinetic-entry comment.
    :return: Category, original site annotation and original reporter suffix.
    :rtype: tuple[str, str | None, str | None]
    """
    explicit = EXPLICIT_SITE_PATTERN.findall(comment)
    relative = RELATIVE_SITE_PATTERN.search(comment)
    reporter = REPORTER_PATTERN.search(substrate)
    if explicit:
        return "reported_bond_or_position", "; ".join(explicit), None
    if relative:
        return "relative_site_label", relative.group(), None
    if reporter:
        return "reporter_construct", None, reporter.group(1)
    return "unresolved", None, None

In [7]:
coverage_rows = []
for row in brenda_kinetic_raw.to_dict("records"):
    raw_value = row["value"] or ""
    match = re.fullmatch(r"\s*([^{}]*?)\s*(?:\{(.*)\})?\s*", raw_value, re.DOTALL)
    value_text = match[1].strip() if match else raw_value.strip()
    substrate = (match[2] or "").strip() if match else ""
    comment = row["comment"] or ""
    status, annotation, reporter = classify_source_entry(substrate, comment)
    point = POINT_PATTERN.fullmatch(value_text)
    numeric_point = bool(point) and all(
        math.isfinite(float(number)) and float(number) >= 0
        for number in point.groups() if number is not None
    )
    coverage_rows.append({
        **row, "substrate_raw": substrate, "site_information_status": status,
        "site_annotation_raw": annotation, "reporter_suffix_raw": reporter,
        "value_field_numeric_point": numeric_point,
    })
brenda_site_audit = pd.DataFrame(coverage_rows)
coverage_summary = (
    brenda_site_audit.groupby("site_information_status")
    .agg(entries=("source_record_id", "size"), numeric_points=("value_field_numeric_point", "sum"))
    .reindex(STATUS_ORDER, fill_value=0)
)
coverage_summary["percent"] = (100 * coverage_summary["entries"] / len(brenda_site_audit)).round(2)
coverage_by_parameter = pd.crosstab(
    brenda_site_audit["parameter_family"], brenda_site_audit["site_information_status"]
).reindex(index=KINETIC_FAMILIES, columns=STATUS_ORDER, fill_value=0)
coverage_by_parameter["total"] = coverage_by_parameter.sum(axis=1)
print("Numeric points count only the original value field; all source entries remain in the counts.")
display(coverage_summary)
display(coverage_by_parameter)

Numeric points count only the original value field; all source entries remain in the counts.


,entries,numeric_points,percent
site_information_status,,,
reported_bond_or_position,192,189,0.72
relative_site_label,12,12,0.04
reporter_construct,7828,7772,29.32
unresolved,18665,17932,69.91


site_information_status,reported_bond_or_position,relative_site_label,reporter_construct,unresolved,total
parameter_family,,,,,
turnover_number,176,4,2983,7476,10639
km_value,16,4,3676,9388,13084
kcat_km_value,0,4,1169,1801,2974


In [8]:
brenda_reported_sites = brenda_site_audit.loc[
    brenda_site_audit["site_information_status"].eq("reported_bond_or_position")
]
print("Examples of original site wording | Matching source entries:", len(brenda_reported_sites))
display(brenda_reported_sites.drop_duplicates("comment")[[
    "source_record_id", "value", "comment", "site_annotation_raw", "proteins", "references"
]].head(PREVIEW_ROWS))
# Inspect further rows directly, for example: display(brenda_reported_sites.iloc[3:6])

Examples of original site wording | Matching source entries: 192


,source_record_id,value,comment,site_annotation_raw,proteins,references
13182,data/3.4.21.6/turnover_number/32,96 {prothrombin},"wild-type substrate, for cleavage at Arg320",cleavage at Arg320,[3],[137]
14277,data/3.4.21.69/km_value/34,0.000102 {Factor VIIIa},"pH 7.2, 37°C, cleavage at Arg336 in A1 subunit",cleavage at Arg336,[2],[40]
21101,data/3.4.23.17/km_value/0,0.0025 {beta-Lipotropin},"human substrate, cleavage at Lys57-Arg58",cleavage at Lys57-Arg58,[4],[2]


## 4. SABIO-RK: raw pages

Read every registered page. The index below is a three-row preview; `sabio_records` contains every original entry.

In [9]:
sabio_pages = {
    filename: json.loads((RAW / "sabio" / filename).read_text())
    for filename in sorted(raw_manifest["sources"]["sabio"]["files"])
}
sabio_records = {}
sabio_locations = {}
for filename, payload in sabio_pages.items():
    for index, record in enumerate(payload["data"]):
        entry_id = record["id"]
        if entry_id in sabio_records:
            raise ValueError(f"Duplicate source entry ID: {entry_id}")
        sabio_records[entry_id] = record
        sabio_locations[entry_id] = {"source_file": filename, "source_index": index}
sabio_raw = pd.DataFrame(sabio_records.values(), dtype=object)
sabio_index = pd.json_normalize(list(sabio_records.values()))[[
    "id", "enzyme_description.ec_number", "enzyme_description.enzyme_name",
    "reaction.equation", "publication.pubmed_id", "publication.title",
]]
print("SABIO-RK entries:", len(sabio_records))
display(pd.DataFrame([page["meta"] for page in sabio_pages.values()]))
display(sabio_index.head(PREVIEW_ROWS))

SABIO-RK entries: 3299


,page,page_size,total_count,total_pages
0,1,1000,3299,4
1,2,1000,3299,4
2,3,1000,3299,4
3,4,1000,3299,4


,id,enzyme_description.ec_number,enzyme_description.enzyme_name,reaction.equation,publication.pubmed_id,publication.title
0,1,3.4.21.62,subtilisin,n+m Peptide + H2O = n Peptide + m Peptide,12962477,Do enzymes change the nature of transition states? Mapping the transition state for general acid-base catalysis of a serine protease
1,2,3.4.21.62,subtilisin,n+m Peptide + H2O = n Peptide + m Peptide,12962477,Do enzymes change the nature of transition states? Mapping the transition state for general acid-base catalysis of a serine protease
2,3,3.4.21.62,subtilisin,n+m Peptide + H2O = n Peptide + m Peptide,12962477,Do enzymes change the nature of transition states? Mapping the transition state for general acid-base catalysis of a serine protease


### Selected entry: reaction and cleavage wording

Show all species of the selected reaction, including substrate/product roles and complete comments. For example, entry `62365` contains `FYPKLQ/ASQAW` in the substrate comment.

In [10]:
sabio_entry = sabio_records[SABIO_ENTRY_ID]
print("Entry:", SABIO_ENTRY_ID, "|", sabio_locations[SABIO_ENTRY_ID])
print("Reaction:", sabio_entry["reaction"].get("equation"))
sabio_species = pd.json_normalize(sabio_entry["reaction"].get("species") or [])
display(sabio_species)

Entry: 62365 | {'source_file': 'page_000002.json', 'source_index': 998}
Reaction: H2O + Peptide = Peptide + Peptide


,stoch_value,role,protein_complex_names,protein_complex_ids,comment,compound.id,compound.name,location.id,location.name,location.go_term
0,1,Catalyst,None,None,purified,0,Enzyme,None,None,None
1,1,Substrate,None,None,-,40,H2O,None,None,None
2,1,Substrate,None,None,FYPKLQ/ASQAW,2562,Peptide,None,None,None
3,1,Product,None,None,-,2562,Peptide,None,None,None
4,1,Product,None,None,-,2562,Peptide,None,None,None


### Parameters, enzyme, conditions and publication

Unpack dictionaries for display. Values, units, normalized fields supplied by SABIO-RK, uncertainties and species references are retained exactly as loaded. Parameter columns correspond to the original zero-based parameter indices.

In [11]:
print("Kinetic parameters")
sabio_parameters = pd.json_normalize(sabio_entry["kineticlaw"].get("parameter") or [])
display(sabio_parameters.T)
for section in ("enzyme_description", "experimental_conditions", "publication"):
    print(section)
    display(pd.DataFrame.from_dict(sabio_entry.get(section) or {}, orient="index", columns=["source_value"]))

Kinetic parameters


,0,1,2,3,4
name,kcat,E,kcat_Km,Km,S
role,Unknown,Unknown,Unknown,Constant,Unknown
start_value,1.57,1.07,2.86,0.549,0.1
end_value,NaN,17.1,NaN,NaN,2.0
standard_deviation,0.28,NaN,0.21,0.105,NaN
comment,-,-,-,-,-
n_start_value,0.026167,0.000001,47.666667,0.000549,0.0001
n_end_value,NaN,0.000017,NaN,NaN,0.002
n_standard_deviation,0.004667,NaN,3.5,0.000105,NaN
parameter_type.id,7,9,15,8,9


enzyme_description


,source_value
ec_number,3.4.22
wildtype,Wildtype
enzyme_name,Cysteine endopeptidases
mutant_spec,None
is_recombinant,True
expressed_in,Escherichia coli
proteins,"[{'stoch_value': None, 'uniprot_id': '(P0C6X7)*2', 'name': None, 'molweight_kda': None, 'molweight_dev': None}]"
go_term,None


experimental_conditions


,source_value
buffer,40 mM Tris-HCl
comment,None
envvar_ph,"{'start_value': 7.3, 'end_value': None}"
envvar_temperature,"{'start_value': 22, 'end_value': None, 'unit': '°C'}"
envvars_other,[]


publication


,source_value
publication_id,6254
pubmed_id,14561748
title,"Biosynthesis, purification, and substrate specificity of severe acute respiratory syndrome coronavirus 3C-like proteinase."
author,"Fan K, Wei P, Feng Q, Chen S, Huang C, Ma L, Lai B, Pei J, Liu Y, Chen J, Lai L"
journal,Biol Chem
volume,279
pages,1637-42
year,2004
data_identifier,None
base_url,None


## 5. Full source fields (optional)

Set `SHOW_FULL_RECORDS = True` to expand every original field of the selected BRENDA EC block and SABIO-RK entry, including empty containers and nulls. The default previews above avoid dumping entire nested records.

In [12]:
def source_fields(value):
    """Show every JSON leaf without interpreting or changing its value.

    :param value: A decoded source object or selected section.
    :return: Paths and original values, including nulls and empty containers.
    :rtype: pandas.DataFrame
    """
    rows = []

    def visit(item, path):
        if isinstance(item, dict) and item:
            for key, child in item.items():
                token = str(key).replace("~", "~0").replace("/", "~1")
                visit(child, f"{path}/{token}")
        elif isinstance(item, list) and item:
            for index, child in enumerate(item):
                visit(child, f"{path}/{index}")
        else:
            rows.append({"source_path": path, "source_value": item})

    visit(value, "")
    return pd.DataFrame(rows, columns=["source_path", "source_value"], dtype=object)

if SHOW_FULL_RECORDS:
    print("BRENDA: complete EC block", BRENDA_EC)
    display(source_fields(brenda_raw))
    print("SABIO-RK: complete entry", SABIO_ENTRY_ID)
    display(source_fields(sabio_entry))